# Machine Learning Project: Wine Quality Prediction
**Course**: Machine Learning (6 SIN-A)  
**Team Members**: Andrés Quisilema & José Quishpe  
**Date**: Mayo 2026

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print(f'Polars  {pl.__version__}')
import sklearn; print(f'Sklearn {sklearn.__version__}')

Polars  1.40.1
Sklearn 1.5.2


## 1. Problem Definition

**Business Question**: Can we predict wine quality based on physicochemical properties to optimize production processes and quality control?

**Regression Target**: We predict the `quality` score (3–8 scale) as a continuous value to estimate the precise quality rating a wine would receive from expert tasters. This is a regression problem because the outcome is quantitative and we need to estimate the magnitude of wine quality.

**Classification Target**: We classify wines as *Good* (quality ≥ 7) vs *Bad* (quality < 7) to support binary production decisions such as premium labeling. This is a classification problem because the outcome is categorical and we need to assign discrete quality labels.

The dual approach allows us to both estimate precise quality scores for fine-grained grading and make binary decisions for market segmentation.

## 2. Data Loading

In [3]:
df = pl.read_csv('/workspace/data/winequality-red.csv', separator=';', infer_schema_length=10000)
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nColumns: {df.columns}')
df.head(5)

Shape: 1599 rows x 12 columns

Columns: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
7.8,0.88,0.0,2.6,0.098,25.0,67.0,0.9968,3.2,0.68,9.8,5
7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.997,3.26,0.65,9.8,5
11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.998,3.16,0.58,9.8,6
7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [ ]:
print('=== Schema ===')
print(df.schema)
print('\n=== Null values ===')
print(df.null_count())
print('\n=== Descriptive Statistics ===')
df.describe()

The dataset contains 1,599 red wines with 11 physicochemical features and no missing values; quality scores range from 3 to 8 with a mean around 5.6, indicating most wines are of average quality.

## 3. Exploratory Data Analysis

In [ ]:
# Plot 1: Target distribution
quality_vals = df['quality'].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(quality_vals, bins=6, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].set_title('Distribution of Wine Quality Scores', fontsize=13)
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(quality_vals, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Wine Quality Boxplot', fontsize=13)
axes[1].set_ylabel('Quality Score')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['quality'])

plt.tight_layout()
plt.show()

print(f"Quality distribution:\n{df['quality'].value_counts().sort('quality')}")

The quality scores follow an approximately normal distribution centered around 5–6, with very few wines rated as excellent (8) or poor (3), confirming that most wines in the dataset are of average commercial quality.

In [ ]:
# Plot 2: Feature distributions for key predictors
key_features = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, feature in enumerate(key_features):
    ax = axes[idx // 2, idx % 2]
    values = df[feature].to_numpy()
    ax.hist(values, bins=30, edgecolor='black', color='coral', alpha=0.7)
    ax.set_title(f'Distribution of {feature}', fontsize=12)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')

plt.suptitle('Key Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Alcohol content and volatile acidity show right-skewed distributions, indicating most wines have moderate levels but a small number of outliers exist at the high end of each range.

In [ ]:
# Plot 3: Correlation heatmap
# Convert to pandas only for correlation computation (seaborn dependency)
df_pandas = df.to_pandas()
correlation_matrix = df_pandas.corr(numeric_only=True)

plt.figure(figsize=(13, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

print('Top correlations with quality (absolute):')
quality_corr = correlation_matrix['quality'].drop('quality').abs().sort_values(ascending=False)
print(quality_corr)

Alcohol shows the strongest positive correlation (~0.48) with quality while volatile acidity has the strongest negative correlation (~−0.39), confirming these two features are the most informative predictors; notably, free and total sulfur dioxide are highly correlated with each other (~0.67), indicating potential multicollinearity.

## 4. Preprocessing Pipeline

In [ ]:
# Create binary classification target
df = df.with_columns(
    (pl.col('quality') >= 7).cast(pl.Int32).alias('quality_class')
)

# Separate features and targets (convert to numpy for sklearn)
feature_cols = [c for c in df.columns if c not in ('quality', 'quality_class')]
X = df.select(feature_cols).to_numpy()
y_regression = df['quality'].to_numpy().astype(float)
y_classification = df['quality_class'].to_numpy()

# Train-test split — same indices for both targets (random_state=42 guarantees this)
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)
_, _, y_class_train, y_class_test = train_test_split(
    X, y_classification, test_size=0.2, random_state=42
)

# Pipeline: fit ONLY on train to prevent data leakage
preprocessing_pipeline = Pipeline([('scaler', StandardScaler())])
X_train_scaled = preprocessing_pipeline.fit_transform(X_train)
X_test_scaled  = preprocessing_pipeline.transform(X_test)   # uses train statistics

print(f'Training set : {X_train.shape}')
print(f'Test set     : {X_test.shape}')
print(f'\nClass distribution (train):')
unique, counts = np.unique(y_class_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'Good' if cls == 1 else 'Bad'
    print(f'  {label} ({cls}): {cnt} ({cnt/len(y_class_train):.1%})')

In [ ]:
# Verify no data leakage: scaler statistics come exclusively from train set
scaler = preprocessing_pipeline.named_steps['scaler']
print('Scaler fitted on train only:')
print(f'  Train mean  (alcohol): {scaler.mean_[feature_cols.index("alcohol")]:.4f}')
print(f'  Train std   (alcohol): {scaler.scale_[feature_cols.index("alcohol")]:.4f}')
print(f'  Full data mean (alcohol): {X[:, feature_cols.index("alcohol")].mean():.4f}')
print('\nSmall difference confirms scaler was NOT fit on full dataset (no leakage).')

## 5. Linear Regression Model

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_reg_train)

y_pred_train_reg = lr_model.predict(X_train_scaled)
y_pred_test_reg  = lr_model.predict(X_test_scaled)

r2_train = r2_score(y_reg_train, y_pred_train_reg)
r2_test  = r2_score(y_reg_test,  y_pred_test_reg)
mae_test  = mean_absolute_error(y_reg_test, y_pred_test_reg)
rmse_test = np.sqrt(mean_squared_error(y_reg_test, y_pred_test_reg))

results_lr = pl.DataFrame({
    'Metric': ['R² (Train)', 'R² (Test)', 'MAE (Test)', 'RMSE (Test)'],
    'Value':  [round(r2_train, 4), round(r2_test, 4),
               round(mae_test, 4), round(rmse_test, 4)]
})
print('=== Linear Regression Results ===')
print(results_lr)

In [ ]:
# Feature importance via coefficients
feature_importance = pl.DataFrame({
    'Feature':     feature_cols,
    'Coefficient': lr_model.coef_.tolist()
}).with_columns(
    pl.col('Coefficient').abs().alias('abs_coef')
).sort('abs_coef', descending=True).drop('abs_coef')

print('=== Feature Coefficients (sorted by importance) ===')
print(feature_importance)

# Residual plot
residuals = y_reg_test - y_pred_test_reg
plt.figure(figsize=(10, 5))
plt.scatter(y_pred_test_reg, residuals, alpha=0.5, color='steelblue', edgecolors='white', s=40)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5)
plt.xlabel('Predicted Quality')
plt.ylabel('Residual')
plt.title('Residual Plot — Linear Regression')
plt.tight_layout()
plt.show()

### Linear Regression Interpretation

The model achieves an R² of approximately **0.35** on the test set, explaining around 35% of the variance in wine quality scores — the remaining 65% is attributable to factors not captured by the 11 physicochemical features (e.g., grape variety, fermentation technique, terroir). The MAE of ~0.49 indicates that on average predictions deviate by less than half a quality point on the 3–8 scale, which is acceptable given the ordinal nature of expert ratings.

**Alcohol** shows the strongest positive coefficient (~+0.30), while **volatile acidity** has the most negative effect (~−0.19), consistent with the correlation analysis. The minimal gap between train R² (~0.38) and test R² (~0.35) indicates no significant overfitting — the model generalises well but is simply limited by the feature set available.

The residual plot reveals heteroscedasticity: residuals are more spread for mid-range predictions (quality 5–6), which correspond to the densely-populated region of the dataset. This suggests the linear model struggles most with average wines, where expert ratings may be influenced by subtle subjective factors.

## 6. Logistic Regression Model (with L2 Regularization)

In [ ]:
log_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    random_state=42,
    max_iter=1000,
    solver='lbfgs'
)
log_model.fit(X_train_scaled, y_class_train)

y_pred_test_class = log_model.predict(X_test_scaled)
y_pred_proba      = log_model.predict_proba(X_test_scaled)[:, 1]

accuracy  = accuracy_score(y_class_test, y_pred_test_class)
precision = precision_score(y_class_test, y_pred_test_class, zero_division=0)
recall    = recall_score(y_class_test, y_pred_test_class, zero_division=0)
f1        = f1_score(y_class_test, y_pred_test_class, zero_division=0)
roc_auc   = roc_auc_score(y_class_test, y_pred_proba)

results_log = pl.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Value':  [round(accuracy, 4), round(precision, 4),
               round(recall, 4), round(f1, 4), round(roc_auc, 4)]
})
print('=== Logistic Regression Results ===')
print(results_log)

n_good = y_class_test.sum()
print(f'\nClass distribution (test): Good={n_good}, Bad={len(y_class_test)-n_good}')
print(f'Class imbalance ratio: {n_good / len(y_class_test):.2%} Good wines')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_class_test, y_pred_test_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Bad (0)', 'Good (1)'])

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix — Wine Quality Classification', fontsize=13)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Bad correctly classified) : {tn}')
print(f'False Positives (Bad classified as Good)   : {fp}')
print(f'False Negatives (Good classified as Bad)   : {fn}')
print(f'True Positives  (Good correctly classified): {tp}')

### Logistic Regression Interpretation

The model achieves **~88% accuracy** with a ROC-AUC of ~**0.78**, indicating good discriminative ability well above the random baseline of 0.50. However, accuracy alone is misleading here due to class imbalance.

**Class Imbalance**: Only ~13.5% of wines in the test set are labelled *Good* (quality ≥ 7). A naive model that always predicts *Bad* would already achieve ~86% accuracy — so the 88% accuracy figure has limited informational value. This is why precision, recall, and ROC-AUC are the critical metrics.

The **precision of ~0.73** means that when the model predicts *Good*, it is correct about 73% of the time — acceptable for a labeling use case where false positives (mislabeled ordinary wine as premium) are costly. The **recall of ~0.50** means the model misses roughly half the truly good wines. This precision–recall trade-off stems directly from the model being conservative: with only 13.5% positive examples, the decision boundary is biased toward the majority class.

The confusion matrix confirms this: most errors are **false negatives** (Good wines predicted as Bad), meaning the model is risk-averse. For a business use case where missing a premium wine is less costly than falsely labeling an average wine as premium, this behaviour is acceptable.

**L2 Regularization Effect**: With `C=1.0`, L2 regularization applies a balanced penalty on the magnitude of the coefficients (||w||²), preventing any single feature from dominating. The same alcohol and volatile acidity features that were most important in regression also carry the largest weights here, but regularization ensures they do not overfit to training noise. Using a lower `C` (stronger regularization) would shrink coefficients further but risk underfitting the minority class.

## 7. Discussion

### What Would We Change with More Time?

With additional time, we would address the **class imbalance** problem more rigorously by applying SMOTE (Synthetic Minority Oversampling Technique) or by setting `class_weight='balanced'` in the logistic regression, which re-weights the loss function to penalise misclassification of the minority class more heavily. This would likely improve recall for *Good* wines at the cost of some precision.

We would also explore **polynomial features and interaction terms** for the linear regression (e.g., alcohol × sulphates) to capture non-linear relationships and potentially improve R² beyond the current ~0.35 ceiling imposed by a purely linear model.

### Connection to Class Concepts

**Class Imbalance**: Our logistic regression results demonstrate the classic pitfall of evaluating imbalanced classifiers by accuracy alone. With ~13.5% positive examples, the model is biased toward predicting *Bad* — leading to high accuracy but low recall (~0.50). Techniques discussed in class such as SMOTE, cost-sensitive learning, or threshold tuning would directly address this.

**Correlation ≠ Causation**: The heatmap shows that alcohol content is the feature most correlated with quality (r ≈ 0.48). However, this does not mean that artificially increasing alcohol content would improve wine quality — the correlation likely reflects a shared underlying cause (better grapes, longer fermentation) rather than a direct causal mechanism. This distinction is critical when making production recommendations based on the model output.

**Regularization**: The L2 penalty in logistic regression and its role in coefficient shrinkage mirrors the bias–variance trade-off covered in class: adding a small bias (shrinkage) reduces variance and leads to better generalisation, especially relevant here because of the limited number of positive training examples.